# Password Generator & Strength Validator
Goal: generate passwords with configurable options, score their strength with several criteria, and compare the results.

In [ ]:
import random
import string
import secrets
import re
import math
import pandas as pd

## 1. Password generator

In [ ]:
def generate_password(length=12, use_uppercase=True, use_digits=True, use_symbols=True):
    pools = [string.ascii_lowercase]
    if use_uppercase:
        pools.append(string.ascii_uppercase)
    if use_digits:
        pools.append(string.digits)
    if use_symbols:
        pools.append(string.punctuation)
    all_chars = "".join(pools)
    chars = [secrets.choice(pool) for pool in pools]
    chars += [secrets.choice(all_chars) for _ in range(length - len(chars))]
    random.SystemRandom().shuffle(chars)
    return "".join(chars)

print(generate_password(12))

## 2. Strength checker

In [ ]:
def check_strength(password):
    classes = [r"[a-z]", r"[A-Z]", r"[0-9]", r"[^A-Za-z0-9]"]
    sizes = [26, 26, 10, 32]
    present = [bool(re.search(c, password)) for c in classes]
    pool = sum(size for size, found in zip(sizes, present) if found)
    entropy = len(password) * math.log2(pool)
    score = (len(password) >= 8) + (len(password) >= 12) + (len(password) >= 16)
    score += sum(present)
    score += (entropy >= 60) + (entropy >= 80)
    if re.search(r"(.)\1\1|(.{2,})\2", password):
        score -= 1
    label = "Strong" if score >= 8 else "Medium" if score >= 5 else "Weak"
    return {"strength_score": score, "strength_label": label, "entropy": round(entropy, 1)}

print(check_strength("aaa123"))
print(check_strength(generate_password(16)))

## 3. Batch of 10 passwords

In [ ]:
configs = [
    (6, False, False, False), (8, False, False, False), (8, True, False, False),
    (8, True, True, False), (10, True, True, True), (12, True, True, False),
    (12, True, True, True), (16, True, True, False), (16, True, True, True),
    (20, True, True, True),
]
results = []
for length, upper, digits, symbols in configs:
    password = generate_password(length, upper, digits, symbols)
    config = f"upper={upper}, digits={digits}, symbols={symbols}"
    results.append({"password": password, "length": length, "configuration": config, **check_strength(password)})
len(results)

## 4. Results table

In [ ]:
df = pd.DataFrame(results)
df.style.background_gradient(subset=["strength_score"], cmap="RdYlGn")

## 5. Weakest and strongest

In [ ]:
def find_weakest(results):
    return min(results, key=lambda r: (r["strength_score"], r["entropy"]))

def find_strongest(results):
    return max(results, key=lambda r: (r["strength_score"], r["entropy"]))

weakest = find_weakest(results)
strongest = find_strongest(results)
print("Weakest:", weakest["password"], "|", weakest["strength_score"], weakest["strength_label"], "|", weakest["configuration"])
print("Strongest:", strongest["password"], "|", strongest["strength_score"], strongest["strength_label"], "|", strongest["configuration"])

## 6. Conclusions and recommendations
- **Weakest:** 6 lowercase letters (score 1, Weak, about 28 bits of entropy). Short passwords with one character type can be guessed quickly.
- **Strongest:** 20 characters with uppercase, digits, and symbols (score 9, Strong, about 131 bits of entropy).
- **Length matters most:** 16 characters with letters and digits (score 8) is stronger than 12 characters with symbols (score 7).
- **Recommendations:** use at least 16 characters, mix uppercase, lowercase, digits, and symbols, avoid repeated characters or patterns, and use the `secrets` module (not `random`) to generate passwords.